# Dis2Vec clustering — common disease maps

Loads the precomputed Dis2Vec clustering table (disease embeddings reduced to
UMAP `x`/`y`) and renders two interactive Plotly maps:

1. **By rarity** — points colored rare vs. common.
2. **By cluster** — points colored by cluster label.

Each map can optionally be saved as a standalone HTML file under `/output`.

In [6]:
# Show which Python interpreter is running this notebook (venv sanity check).
import sys
print(sys.executable)

/Users/kwfh644/GitHub/dis2vec-public/.venv/bin/python


In [7]:
# Core data library.
import pandas as pd

In [8]:
# Plotting: Plotly Express for the interactive UMAP scatter maps.
import plotly.express as px

In [9]:
def plot_res(df, title_postfix, color, annotate_orph=None, save=False,
             custom_header=None, postfix_file=''):
    """Render an interactive UMAP scatter of the disease embeddings.

    df            : dataframe with UMAP coords in 'x'/'y' plus metadata columns.
    color         : column used to color the points (e.g. 'rare', 'clust lbl').
    annotate_orph : {disease_id: {ax, ay, dx, dy, name}} arrow annotations to add.
    save          : if True, also write a standalone HTML to /.
    postfix_file  : suffix for the saved HTML filename.
    """
    # Use the caller-supplied header when given, otherwise the axis label.
    title = custom_header if custom_header is not None else title_postfix

    fig = px.scatter(
        df,
        x="x", y="y",
        color=color,
        opacity=0.7,
        hover_data={
            'x': False,
            'y': False,
            'name': True,
            'disease_id': True,
            'dis_vec_clust': True,
        },
        log_x=False, size_max=40, height=550,
        title=title,
    )

    # White background with mirrored, ticked black axes labeled as UMAP dimensions.
    fig.update_layout(
        xaxis_title="UMAP1",
        yaxis_title="UMAP2",
        plot_bgcolor='white',
    )
    fig.update_xaxes(mirror=True, ticks='outside', linecolor='black', dtick=2)
    fig.update_yaxes(mirror=True, ticks='outside', linecolor='black', dtick=2)

    if annotate_orph is None:
        annotate_orph = {}

    # Drop any annotation keys flagged with '~' (disabled entries).
    for ann_items in annotate_orph.values():
        for annot_key in list(ann_items.keys()):
            if '~' in annot_key:
                del ann_items[annot_key]

    # Add an arrow annotation for each disease_id present in both df and annotate_orph.
    for _, r in df[df['disease_id'].isin(annotate_orph.keys())].iterrows():
        annot_item = annotate_orph[r['disease_id']]

        # Label with a custom 'name' if provided, else the disease's own name.
        annot_text = annot_item.get('name', r['name'])

        # Optional offsets (dx/dy) shift the arrow head off the exact marker.
        dx = annot_item.pop('dx', 0)
        dy = annot_item.pop('dy', 0)

        fig.add_annotation(
            xref="x",
            yref="y",
            x=r['x'] + dx,
            y=r['y'] + dy,
            text=annot_text,
            arrowhead=2,
            **annot_item,
        )

    if save is True:
        path_save = "output/Dis2Vec_map_{}.html".format(postfix_file)
        fig.write_html(path_save)
        print(path_save)

    fig.show()

In [10]:
# Load the precomputed clustering table: disease embeddings reduced to UMAP x/y,
# with cluster labels ('dis_vec_clust' / 'clust lbl') and rarity ('rare').
dis2vec_clust_df = pd.read_excel('../data/dis2vec_clust.xlsx')

In [11]:
# --- Map 1: color points by rarity (rare vs. common) ---
# Sort by 'rare' so points draw in a consistent order.
dis2vec_clust_df = dis2vec_clust_df.sort_values(by='rare')

# Arrow annotations: {disease_id: {ax, ay label offsets}}.
annot = {
    'EFO_0000756': {'ay': 30, 'ax': -120},
    'EFO_0001663': {'ay': 60, 'ax': -100},
    'ORPHA_157794': {'ay': -100, 'ax': 120},
    'ORPHA_524': {'ay': -75, 'ax': -90},

    'MONDO_0005180': {'ay': -70, 'ax': 200},
    'EFO_0003108': {'ay': -90, 'ax': 150},
    'ORPHA_370103': {'ay': -90, 'ax': -200},
    'ORPHA_412217': {'ay': -80, 'ax': -10},
    'ORPHA_238455': {'ay': -110, 'ax': -120},
}

plot_res(
    dis2vec_clust_df, 'UMAP',
    annotate_orph=annot,
    color='rare',
    custom_header='',
    save=True,
    postfix_file='rare_UMAP',
)

output/Dis2Vec_map_rare_UMAP.html


In [12]:
# --- Map 2: color points by cluster label ---
# Sort by numeric cluster id so the color sequence is stable.
dis2vec_clust_df = dis2vec_clust_df.sort_values(by='dis_vec_clust')

# Highlight a few example diseases (e.g. LALD/CESD, NF1/NF2 groups).
annot = {
    'ORPHA_275761': {'ay': -60, 'ax': -60},
    'ORPHA_75234': {'ay': -100, 'ax': +60},
    'ORPHA_75233': {'ay': -35, 'ax': +35},
    'ORPHA_637': {'ay': -50, 'ax': -20},
    'ORPHA_636': {'ay': -30, 'ax': -20},
}

plot_res(
    dis2vec_clust_df, 'UMAP',
    annotate_orph=annot,
    color='clust lbl',
    custom_header='',
    save=True,
    postfix_file='dis_vec_clust_UMAP',
)

output/Dis2Vec_map_dis_vec_clust_UMAP.html
